<a href="https://colab.research.google.com/github/shuchiisharmaa/Audio-Detection-MLP/blob/main/Audio_Detection_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# CELL 1
!pip install librosa soundfile numpy scikit-learn matplotlib seaborn tqdm --quiet
print("Setup complete")

ERROR: Operation cancelled by user
^C


In [ ]:
# CELL 2
import os
import warnings
import numpy as np
import pandas as pd
import librosa
import librosa.display
import soundfile as sf
import matplotlib.pyplot as plt
from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

warnings.filterwarnings("ignore")
print("Libraries loaded")

In [ ]:
# CELL 3
def add_noise(y):
    noise = 0.003 * np.random.randn(len(y))
    return y + noise

def stretch(y):
    rate = np.random.uniform(0.85, 1.15)
    return librosa.effects.time_stretch(y, rate=rate)

def shift_pitch(y, sr):
    steps = np.random.uniform(-2, 2)
    return librosa.effects.pitch_shift(y, sr=sr, n_steps=steps)

print("Augmentation ready")

In [ ]:
# CELL 4
def extract_features_from_signal(y, sr):
    y, _ = librosa.effects.trim(y)

    if len(y) < sr:
        y = np.pad(y, (0, sr - len(y)))

    features = []

    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40)
    features.extend(np.mean(mfcc.T, axis=0))
    features.extend(np.std(mfcc.T, axis=0))

    delta = librosa.feature.delta(mfcc)
    features.extend(np.mean(delta.T, axis=0))

    chroma = librosa.feature.chroma_stft(y=y, sr=sr)
    features.extend(np.mean(chroma.T, axis=0))

    mel = librosa.feature.melspectrogram(y=y, sr=sr)
    features.extend(np.mean(mel.T, axis=0)[:40])

    contrast = librosa.feature.spectral_contrast(y=y, sr=sr)
    features.extend(np.mean(contrast.T, axis=0))

    f0, _, _ = librosa.pyin(y, fmin=75, fmax=600)
    features.append(np.nanmean(f0))
    features.append(np.nanstd(f0))

    return np.nan_to_num(np.array(features))

def extract_features(file_path):
    y, sr = librosa.load(file_path, sr=22050, duration=3)
    return extract_features_from_signal(y, sr)

print("Feature extractor ready")

In [ ]:
# CELL 5
!wget -q https://zenodo.org/record/1188976/files/Audio_Speech_Actors_01-24.zip
!unzip -qo Audio_Speech_Actors_01-24.zip -d ravdess
print("Dataset downloaded")

In [ ]:
# CELL 6
emotion_map = {
    '01': 'neutral',
    '02': 'calm',
    '03': 'happy',
    '04': 'sad',
    '05': 'angry',
    '06': 'fearful',
    '07': 'disgust',
    '08': 'surprised'
}

X = []
y = []

files = []
for root, dirs, filenames in os.walk("ravdess"):
    for f in filenames:
        if f.endswith(".wav"):
            files.append(os.path.join(root, f))

print("Processing files...")

for file in tqdm(files):
    fname = os.path.basename(file)
    emotion_code = fname.split("-")[2]
    emotion = emotion_map[emotion_code]

    try:
        audio, sr = librosa.load(file, sr=22050, duration=3)

        X.append(extract_features_from_signal(audio, sr))
        y.append(emotion)

        X.append(extract_features_from_signal(add_noise(audio), sr))
        y.append(emotion)

        X.append(extract_features_from_signal(shift_pitch(audio, sr), sr))
        y.append(emotion)

    except:
        continue

X = np.array(X)
y = np.array(y)

print("Dataset shape:", X.shape)
print("Classes:", np.unique(y))

In [ ]:
# CELL 7
plt.figure(figsize=(10,5))
unique, counts = np.unique(y, return_counts=True)
plt.bar(unique, counts)
plt.title("Emotion Distribution")
plt.xlabel("Emotion")
plt.ylabel("Samples")
plt.show()

In [ ]:
# CELL 8
le = LabelEncoder()
y_encoded = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

In [ ]:
# CELL 9
model = RandomForestClassifier(
    n_estimators=500,
    max_depth=25,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

print("Training complete")

In [ ]:
pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred))
print()
print(classification_report(y_test, pred, target_names=le.classes_))

cm = confusion_matrix(y_test, pred)

plt.figure(figsize=(9,7))
plt.imshow(cm)
plt.xticks(range(len(le.classes_)), le.classes_, rotation=45)
plt.yticks(range(len(le.classes_)), le.classes_)
plt.title("Confusion Matrix")
plt.colorbar()
plt.show()

In [ ]:
# CELL 12 (FULL REPLACEMENT)

from google.colab import files
from IPython.display import Audio, display
import librosa
import numpy as np
import matplotlib.pyplot as plt

print("Upload MP3 / WAV")
uploaded = files.upload()
filename = list(uploaded.keys())[0]

audio, sr = librosa.load(filename, sr=22050)
display(Audio(audio, rate=sr))


# -----------------------------
# detect sound type
# -----------------------------
def detect_audio_type(y, sr):

    centroid = np.mean(
        librosa.feature.spectral_centroid(
            y=y,
            sr=sr
        )
    )

    bandwidth = np.mean(
        librosa.feature.spectral_bandwidth(
            y=y,
            sr=sr
        )
    )

    zcr = np.mean(
        librosa.feature.zero_crossing_rate(
            y
        )
    )

    rms = np.mean(
        librosa.feature.rms(
            y=y
        )
    )

    mfcc = librosa.feature.mfcc(
        y=y,
        sr=sr,
        n_mfcc=13
    )

    mfcc_var = np.mean(
        np.var(mfcc, axis=1)
    )

    # crowd / applause / noisy burst
    if zcr > 0.12 and bandwidth > 2500:
        return "crowd/noise"

    # music
    if centroid > 2500 and mfcc_var > 800:
        return "music"

    # laughter
    if 0.06 < zcr < 0.16 and rms > 0.03 and mfcc_var > 400:
        return "laughter"

    return "speech"


# -----------------------------
# analyze chunks
# -----------------------------
chunk_sec = 3
chunk_size = sr * chunk_sec

timeline = []
times = []
confidences = []

for i in range(0, len(audio), chunk_size):

    chunk = audio[i:i+chunk_size]

    if len(chunk) < sr:
        continue

    t = i / sr

    kind = detect_audio_type(
        chunk,
        sr
    )

    # -----------------------------
    # NON-SPEECH CLASSES
    # -----------------------------
    if kind != "speech":

        centroid = np.mean(
            librosa.feature.spectral_centroid(
                y=chunk,
                sr=sr
            )
        )

        bandwidth = np.mean(
            librosa.feature.spectral_bandwidth(
                y=chunk,
                sr=sr
            )
        )

        zcr = np.mean(
            librosa.feature.zero_crossing_rate(
                chunk
            )
        )

        rms = np.mean(
            librosa.feature.rms(
                y=chunk
            )
        )

        score = (
            min(zcr * 300, 30) +
            min(rms * 700, 30) +
            min(bandwidth / 120, 40)
        )

        score = min(score, 95)

        timeline.append(kind)
        confidences.append(score)
        times.append(t)

        continue

    # -----------------------------
    # SPEECH EMOTION MODEL
    # -----------------------------
    feat = extract_features_from_signal(
        chunk,
        sr
    ).reshape(1, -1)

    feat = scaler.transform(
        feat
    )

    probs = model.predict_proba(
        feat
    )[0]

    idx = np.argmax(
        probs
    )

    emotion = le.classes_[idx]
    conf = probs[idx] * 100

    timeline.append(emotion)
    confidences.append(conf)
    times.append(t)


# -----------------------------
# print results
# -----------------------------
print("\nAudio Timeline:\n")

for t, e, c in zip(
    times,
    timeline,
    confidences
):

    print(
        f"{int(t):02d}s -> {e.upper()} ({c:.1f}%)"
    )


valid = [
    x for x in timeline
    if x in le.classes_
]

if valid:

    overall = max(
        set(valid),
        key=valid.count
    )

    print(
        f"\nOverall dominant emotion: {overall.upper()}"
    )

else:

    dominant = max(
        set(timeline),
        key=timeline.count
    )

    print(
        f"\nDominant audio type: {dominant.upper()}"
    )


# -----------------------------
# visualization
# -----------------------------
plt.figure(
    figsize=(15,5)
)

plt.plot(
    times,
    confidences,
    linewidth=3
)

plt.scatter(
    times,
    confidences,
    s=120
)

for t, e, c in zip(
    times,
    timeline,
    confidences
):

    plt.text(
        t,
        c + 2,
        e,
        ha='center',
        fontsize=11
    )

plt.title(
    "Audio Emotion Timeline",
    fontsize=18
)

plt.xlabel(
    "Time (seconds)",
    fontsize=13
)

plt.ylabel(
    "Confidence (%)",
    fontsize=13
)

plt.ylim(
    0,
    105
)

plt.grid(
    alpha=.3
)

plt.show()